### Croma embeddings for each case of cloud filtering

This notebook:

Create the optical embeddings for the three different cases:
1. v1: QA60+SCL
2. v2: CS+ no dilation
3. v3: CS+ dilated

In [ ]:
import os 
import glob
import numpy as np
import pandas as pd
import geopandas as gpd
import torch
import rasterio
import rasterio.features
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPolygon
from pathlib import Path
from sklearn.decomposition import PCA
from torchgeo.models import croma_base, CROMABase_Weights

# ── Configuration ──
SCRIPT_DIR = os.path.dirname(os.path.abspath("__file__"))

if os.path.exists(os.path.expanduser("~/thesis_tiles_120px")):
    TILES_DIR = os.path.expanduser("~/thesis_tiles_120px")
    SHP_DIR = os.path.join(SCRIPT_DIR, "data_shp")
    DATA_DIR = os.path.join(SCRIPT_DIR, "data_csv")
else:
    TILES_DIR = "/Users/angelicamariamorenorojas/Desktop/Master/thesis/tiles_120px"
    SHP_DIR = os.path.join(SCRIPT_DIR, "data_shp")
    DATA_DIR = os.path.join(SCRIPT_DIR, "data_csv")

# CROMA config
PATCH_SIZE = 8       # 8x8 pixel patches
EMBED_DIM = 768      # ViT-B embedding dimension
TILE_SIZE = 120      # native CROMA tile size (multiple of 8, no cropping)
GRID_SIZE = TILE_SIZE // PATCH_SIZE  # 15x15 tokens

# S2 L2A bands in download order -> CROMA expects 12 bands (no B10)
CROMA_S2_BAND_INDICES = list(range(12))  # all 12 bands

# Device
device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available()
                      else "cpu")
print(f"Device: {device}")
print(f"Tiles dir: {TILES_DIR}")
print(f"CSV dir:   {DATA_DIR}")
print(f"CROMA: {TILE_SIZE}x{TILE_SIZE} -> {GRID_SIZE}x{GRID_SIZE} patch grid -> {EMBED_DIM}-dim tokens")

## 1. Helper functions: loading, normalization, cropping

In [4]:
def normalize(x):
    """Per-channel robust normalization to [0, 1] (same as CROMA repository)."""
    x = x.float()
    imgs = []
    for ch in range(x.shape[1]):
        channel = x[:, ch, :, :]
        min_val = channel.mean() - 2 * channel.std()
        max_val = channel.mean() + 2 * channel.std()
        img = (channel - min_val) / (max_val - min_val + 1e-10)
        img = torch.clamp(img, 0, 1)
        imgs.append(img.unsqueeze(1))
    return torch.cat(imgs, dim=1)


def load_tile(tif_path, band_indices=None):
    """Load a 120x120 native-CROMA tile and return data, transform, crs."""
    with rasterio.open(tif_path) as src:
        data = src.read().astype(np.float32)
        transform = src.transform
        crs = src.crs
    if band_indices is not None:
        data = data[band_indices]
    return data, transform, crs

#Find the sample tile for a given fid 
def find_sample_tile(fid, product, window="evt"):
    pattern = os.path.join(TILES_DIR, product, f"fid_{fid}", window, "*", "tile_0.tif")
    tifs = sorted(glob.glob(pattern))
    return tifs[0] if tifs else None

#List all the fids with both S2 L2A and S1 GRD
def list_available_fids():
    s2_dir = os.path.join(TILES_DIR, "s2_l2a")
    s1_dir = os.path.join(TILES_DIR, "s1_grd")
    if not os.path.exists(s2_dir) or not os.path.exists(s1_dir):
        return []
    s2_fids = {d.replace("fid_", "") for d in os.listdir(s2_dir) if d.startswith("fid_")}
    s1_fids = {d.replace("fid_", "") for d in os.listdir(s1_dir) if d.startswith("fid_")}
    return sorted(s2_fids & s1_fids)

available_fids = list_available_fids()
print(f"FIDs with both S2 L2A and S1 GRD: {len(available_fids)}")
if available_fids:
    print(f"Examples: {available_fids[:10]}")
print(f"\nSingle forward pass: {TILE_SIZE}x{TILE_SIZE} -> {GRID_SIZE}x{GRID_SIZE} tokens (no sliding window needed)")

FIDs with both S2 L2A and S1 GRD: 41
Examples: ['101', '112', '113', '115', '116', '125', '13', '153', '156', '161']

Single forward pass: 120x120 -> 15x15 tokens (no sliding window needed)


In [5]:
def load_croma_model(modalities):
    model = croma_base(
        weights=CROMABase_Weights.CROMA_VIT,
        modalities=modalities,
        image_size=TILE_SIZE,
    )
    model = model.to(device)
    model.eval()
    return model

In [ ]:
# Load the three cloud-filter scenarios. Each CSV is an independent search
# (NOT a nested subset): the same FID may have different evt/bef image_ids
# per version because each filter accepts/rejects a different set of S2 scenes.
scenarios = {
    'v1': pd.read_csv(os.path.join(DATA_DIR, 'v1_images_s2_s1.csv')),
    'v2': pd.read_csv(os.path.join(DATA_DIR, 'v2_images_s2_s1.csv')),
    'v3': pd.read_csv(os.path.join(DATA_DIR, 'v3_images_s2_s1.csv')),
}

scenario_labels = {
    'v1': 'QA60+SCL',
    'v2': 'CS+ no dilation',
    'v3': 'CS+ dilated',
}

# Polygon ground truth (used for both visual overlay and inside-polygon mask)
shp_path = os.path.join(SHP_DIR, "label_polygons.shp")
gdf = gpd.read_file(shp_path)
gdf["fid"] = gdf["fid"].astype(int).astype(str)


def get_image_ids(scenario_df, fid, window):
    """S2 image_ids in CSV for (fid, window). Empty list if missing/NaN."""
    col = f"{window}IdsS2"
    row = scenario_df[scenario_df["fid"].astype(int) == int(fid)]
    if row.empty:
        return []
    val = row[col].iloc[0]
    if pd.isna(val) or not isinstance(val, str) or not val.strip():
        return []
    return [s.strip() for s in val.split(",") if s.strip()]


def find_tile_for_image(fid, product, window, image_id, tile_idx=0):
    """Locate a specific tile on disk for (fid, window, image_id)."""
    path = os.path.join(TILES_DIR, product, f"fid_{fid}", window, image_id, f"tile_{tile_idx}.tif")
    return path if os.path.exists(path) else None


def polygon_mask_on_grid(fid, transform, crs, grid_size=GRID_SIZE, patch_size=PATCH_SIZE):
    """Rasterize polygon to (grid_size, grid_size) bool mask. One cell = one CROMA patch."""
    poly_row = gdf[gdf["fid"] == str(fid)]
    if poly_row.empty:
        return np.zeros((grid_size, grid_size), dtype=bool)
    poly_reproj = poly_row.to_crs(crs)
    px = transform.a * patch_size           # patch resolution in CRS units (80 m)
    py = -transform.e * patch_size
    patch_transform = rasterio.transform.from_origin(transform.c, transform.f, px, py)
    geoms = [(g, 1) for g in poly_reproj.geometry if not g.is_empty]
    mask = rasterio.features.rasterize(
        geoms, out_shape=(grid_size, grid_size),
        transform=patch_transform, fill=0, all_touched=True, dtype="uint8"
    ).astype(bool)
    return mask


print(f"Loaded {len(gdf)} polygons")
for name, df in scenarios.items():
    print(f"  {name} ({scenario_labels[name]}): {len(df)} rows")

## 2. Find FIDs where versions disagree

A meaningful comparison needs FIDs where v1/v2/v3 actually picked **different**
images. If all three filters return the same image_id, the embeddings are identical
and the comparison is empty. We also require that `tile_0` exists on disk for every
(version × window), otherwise the run skips the FID.

In [ ]:
def fids_with_divergent_versions(scenarios, windows=("evt", "bef"), check_disk=True):
    """FIDs where the first(image_id) differs across versions for at least one window.
    If check_disk=True, only keep FIDs where every (version, window) has tile_0 on disk
    (so the comparison is actually runnable).
    Returns: list of {'fid', 'per_version': {v: {window: image_id}}, 'differ_in': [windows...]}
    """
    all_fids = sorted({int(f) for df in scenarios.values() for f in df["fid"]})
    out = []
    for fid in all_fids:
        per_version = {}
        runnable = True
        for v_name, df in scenarios.items():
            entry = {}
            for w in windows:
                ids = get_image_ids(df, fid, w)
                if not ids:
                    entry[w] = None
                    runnable = False
                    continue
                image_id = ids[0]
                if check_disk and find_tile_for_image(fid, "s2_l2a", w, image_id) is None:
                    runnable = False
                entry[w] = image_id
            per_version[v_name] = entry
        if check_disk and not runnable:
            continue
        differ_in = [w for w in windows
                     if len({per_version[v][w] for v in per_version
                             if per_version[v][w] is not None}) > 1]
        if differ_in:
            out.append({"fid": fid, "per_version": per_version, "differ_in": differ_in})
    return out


divergent = fids_with_divergent_versions(scenarios, windows=("evt", "bef"), check_disk=True)
print(f"FIDs where v1/v2/v3 picked DIFFERENT images (and all tiles on disk): {len(divergent)}\n")

# Compact summary: one row per FID, marking which windows disagree
for entry in divergent:
    flags = ", ".join(f"{w} differs" for w in entry["differ_in"])
    print(f"  FID {entry['fid']:>4}  —  {flags}")

# Detail of the first divergent FID
if divergent:
    fid = divergent[0]["fid"]
    print(f"\nDetail of FID {fid}:")
    for v, entry in divergent[0]["per_version"].items():
        for w in ("evt", "bef"):
            print(f"  {v} ({scenario_labels[v]:<18}) {w}: {entry[w]}")
else:
    print("No divergent FIDs found with tiles on disk. "
          "Either disk coverage is incomplete or all filters agreed on every FID.")

## 3. Extract embeddings per version on a divergent FID

We pick a FID from the divergent list (where v1/v2/v3 picked different images), look
up the evt/bef `image_id` per version, locate `tile_0` on disk in the unified
download directory, run CROMA's optical encoder, and build the change map
`1 − cosine_sim(evt, bef)` on the 15×15 patch grid.

To use a specific FID instead of the auto-pick, set `TEST_FID = <number>` manually.

In [ ]:
# ── Encoder + change-map helpers ──
def encode_tile(model, tile_data, modality):
    """Run CROMA on a 120x120 tile, return (15, 15, 768) patch tokens."""
    x = torch.from_numpy(tile_data).unsqueeze(0)
    x = normalize(x).to(device)

    captured = {}
    encoder = model.s2_encoder if modality == "optical" else model.s1_encoder

    def hook(module, inp, out):
        captured['tokens'] = out

    h = encoder.register_forward_hook(hook)
    with torch.no_grad():
        if modality == "optical":
            _ = model(x_optical=x)
        else:
            _ = model(x_sar=x)
    h.remove()

    out = captured['tokens']
    if isinstance(out, tuple):
        out = out[0]
    tokens = out.cpu().numpy().squeeze()
    expected = GRID_SIZE * GRID_SIZE
    if tokens.shape[0] == expected + 1:
        tokens = tokens[1:]
    return tokens.reshape(GRID_SIZE, GRID_SIZE, EMBED_DIM)


def cosine_change_map(tokens_a, tokens_b):
    """Per-patch (1 - cosine_similarity) between two token grids."""
    h, w = tokens_a.shape[:2]
    a = tokens_a.reshape(-1, EMBED_DIM)
    b = tokens_b.reshape(-1, EMBED_DIM)
    a_norm = a / (np.linalg.norm(a, axis=1, keepdims=True) + 1e-10)
    b_norm = b / (np.linalg.norm(b, axis=1, keepdims=True) + 1e-10)
    sim = (a_norm * b_norm).sum(axis=1).reshape(h, w)
    return 1.0 - sim


# ── Pick a FID where v1/v2/v3 chose different images (auto from section 2) ──
# Override manually if you want a specific FID, e.g. TEST_FID = 140
TEST_FID = divergent[0]["fid"] if divergent else 140
print(f"Test FID: {TEST_FID}\n")

for name, df in scenarios.items():
    evt_ids = get_image_ids(df, TEST_FID, "evt")
    bef_ids = get_image_ids(df, TEST_FID, "bef")
    print(f"  {name} ({scenario_labels[name]}):")
    print(f"     evt: {evt_ids if evt_ids else '(empty)'}")
    print(f"     bef: {bef_ids if bef_ids else '(empty)'}")

# ── Extract optical embeddings per version (first image_id of each window) ──
print("\nLoading optical model...")
model_opt = load_croma_model(['optical'])

results = {}
for name, df in scenarios.items():
    evt_ids = get_image_ids(df, TEST_FID, "evt")
    bef_ids = get_image_ids(df, TEST_FID, "bef")
    if not evt_ids or not bef_ids:
        print(f"  {name}: skip (missing evt or bef in CSV)")
        results[name] = None
        continue
    evt_id, bef_id = evt_ids[0], bef_ids[0]
    evt_path = find_tile_for_image(TEST_FID, "s2_l2a", "evt", evt_id)
    bef_path = find_tile_for_image(TEST_FID, "s2_l2a", "bef", bef_id)
    if evt_path is None or bef_path is None:
        print(f"  {name}: tile_0 not on disk (evt={evt_path is not None}, bef={bef_path is not None})")
        results[name] = None
        continue
    evt_data, evt_tr, evt_crs = load_tile(evt_path, band_indices=CROMA_S2_BAND_INDICES)
    bef_data, _, _ = load_tile(bef_path, band_indices=CROMA_S2_BAND_INDICES)
    evt_tokens = encode_tile(model_opt, evt_data, "optical")
    bef_tokens = encode_tile(model_opt, bef_data, "optical")
    change = cosine_change_map(evt_tokens, bef_tokens)
    results[name] = {
        "evt_id": evt_id, "bef_id": bef_id,
        "evt_data": evt_data, "bef_data": bef_data,
        "evt_tokens": evt_tokens, "bef_tokens": bef_tokens,
        "change": change, "transform": evt_tr, "crs": evt_crs,
    }
    print(f"  {name}: change shape={change.shape}, "
          f"mean={change.mean():.3f}, min={change.min():.3f}, max={change.max():.3f}")

del model_opt
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 4. Inside-polygon metric

We rasterize the labeled polygon onto the 15×15 patch grid (cell = 80 m × 80 m).
For each version we report `mean(change_inside)`: how much CROMA "sees" the labeled
disturbance. Higher = the filter preserved enough usable data to detect the event.

We **don't** compare against "outside" because outside the polygon is not ground-truth
"no change" — there can be unlabeled disturbance, river/seasonal effects, etc.

In [ ]:
print(f"FID {TEST_FID} — inside-polygon change (higher = stronger detection)\n")
print(f"{'Version':<6} {'Filter':<22} {'patches_in':>11} {'mean(in)':>10} {'median(in)':>11} {'mean(all)':>11}")
print("-" * 75)
for name, res in results.items():
    if res is None:
        print(f"{name:<6} {scenario_labels[name]:<22} {'(skipped)':>11}")
        continue
    mask = polygon_mask_on_grid(TEST_FID, res["transform"], res["crs"])
    inside = res["change"][mask]
    n_in = int(mask.sum())
    if n_in == 0:
        print(f"{name:<6} {scenario_labels[name]:<22} {n_in:>11} {'(no patches inside polygon)':>10}")
        continue
    mean_in = float(inside.mean())
    med_in = float(np.median(inside))
    mean_all = float(res["change"].mean())
    print(f"{name:<6} {scenario_labels[name]:<22} {n_in:>11} "
          f"{mean_in:>10.4f} {med_in:>11.4f} {mean_all:>11.4f}")

## 5. Side-by-side comparison

Three columns (v1/v2/v3), four rows:
1. RGB of the **event** scene used by that version
2. RGB of the **before** scene used by that version
3. Per-patch **change map** (`hot` cmap, 0–0.5)
4. Polygon mask on the patch grid (white = inside polygon → cells used for `mean(in)`)

In [ ]:
def _polygon_pixel_coords(fid, transform, crs):
    """Polygon outline in pixel coordinates of a given transform."""
    poly_row = gdf[gdf["fid"] == str(fid)]
    if poly_row.empty:
        return []
    poly_reproj = poly_row.to_crs(crs)
    coords_list = []
    for geom in poly_reproj.geometry:
        if geom.is_empty:
            continue
        geoms = [geom] if geom.geom_type == "Polygon" else list(geom.geoms)
        for g in geoms:
            xs, ys = g.exterior.coords.xy
            coords_list.append([(~transform * (x, y)) for x, y in zip(xs, ys)])
    return coords_list


def add_polygon_overlay(ax, fid, transform, crs, scale_factor=1):
    """Outline polygon on an image. scale_factor=PATCH_SIZE for the 15x15 grid."""
    for coords in _polygon_pixel_coords(fid, transform, crs):
        scaled = [(x / scale_factor, y / scale_factor) for x, y in coords]
        ax.add_patch(MplPolygon(scaled, closed=True,
                                edgecolor="red", facecolor="none", linewidth=2))


valid = [(name, res) for name, res in results.items() if res is not None]
n_v = len(valid)
if n_v == 0:
    print("No version produced a usable result for this FID.")
else:
    fig, axes = plt.subplots(4, n_v, figsize=(5 * n_v, 18))
    if n_v == 1:
        axes = axes[:, np.newaxis]

    for col, (name, res) in enumerate(valid):
        # Row 0: EVT RGB
        ax = axes[0, col]
        rgb = res["evt_data"][[3, 2, 1]].transpose(1, 2, 0)
        rgb = np.clip(rgb / 3000.0, 0, 1)
        ax.imshow(rgb)
        add_polygon_overlay(ax, TEST_FID, res["transform"], res["crs"], 1)
        ax.set_title(f"{name} ({scenario_labels[name]})\nEVT RGB — {res['evt_id'][:15]}…", fontsize=9)
        ax.axis("off")

        # Row 1: BEF RGB
        ax = axes[1, col]
        rgb = res["bef_data"][[3, 2, 1]].transpose(1, 2, 0)
        rgb = np.clip(rgb / 3000.0, 0, 1)
        ax.imshow(rgb)
        add_polygon_overlay(ax, TEST_FID, res["transform"], res["crs"], 1)
        ax.set_title(f"BEF RGB — {res['bef_id'][:15]}…", fontsize=9)
        ax.axis("off")

        # Row 2: change map (15x15)
        ax = axes[2, col]
        im = ax.imshow(res["change"], cmap="hot", vmin=0, vmax=0.5, interpolation="nearest")
        add_polygon_overlay(ax, TEST_FID, res["transform"], res["crs"], PATCH_SIZE)
        ax.set_title("Change (1 - cos_sim)", fontsize=9)
        ax.axis("off")
        plt.colorbar(im, ax=ax, fraction=0.046)

        # Row 3: polygon mask used for mean(in)
        ax = axes[3, col]
        mask = polygon_mask_on_grid(TEST_FID, res["transform"], res["crs"])
        ax.imshow(mask, cmap="gray", vmin=0, vmax=1, interpolation="nearest")
        add_polygon_overlay(ax, TEST_FID, res["transform"], res["crs"], PATCH_SIZE)
        ax.set_title(f"Polygon mask on patch grid ({int(mask.sum())} patches)", fontsize=9)
        ax.axis("off")

    fig.suptitle(f"FID {TEST_FID} — Cloud-filter comparison (CROMA optical)", fontsize=12, y=1.01)
    plt.tight_layout()
    plt.show()